# Delta Lake MERGE Implementation — Incremental Data Processing (SCD Type 1 & Type 2)

**Objective:** Perform incremental data processing using Delta Lake's `MERGE INTO` semantics.

**Dataset:** A `customer_master` table (target) and a `customer_incremental` table (source),
derived from the Superstore dataset's customer dimension.

**Steps covered:**
1. Load the dataset into a Delta table
2. Perform basic cleaning (handle nulls, remove duplicates)
3. Create a second dataset simulating new/incremental data
4. Apply a MERGE operation to update existing records and insert new ones (SCD Type 1)
5. Apply a MERGE operation that preserves history (SCD Type 2)
6. Validate results (row count, duplicates)
7. Display the final dataset and summary


In [1]:
pip install pandas numpy pyarrow jupyter ipykernel

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 79.5/79.5 kB 543.6 kB/s eta 0:00:00a 0:00:01
  Using cached pyarrow-24.0.0-cp312-cp312-manylinux_2_28_x86_64.whl.metadata (3.0 kB)
  Using cached jupyter-1.1.1-py2.py3-none-any.whl.metadata (2.0 kB)
  Using cached jupyter_console-6.6.3-py3-none-any.whl.metadata (5.8 kB)
  Using cached nbconvert-7.17.1-py3-none-any.whl.metadata (8.4 kB)
  Using cached httpx-0.28.1-py3-none-any.whl.metadata (7.1 kB)
  Using cached jinja2-3.1.6-py3-none-any.whl.metadata (2.9 kB)
  Using cached notebook_shim-0.2.4-py3-none-any.whl.metadata (4.0 kB)
  Using cached defusedxml-0.7.1-py2.py3-none-any.whl.metadata (32 kB)
  Using cached jupyterlab_pygments-0.3.0-py3-none-any.whl.metadata (4.4 kB)
  Using cached markupsafe-3.0.3-cp312-cp312-manylinux2014_x86_64.manylinux_2_17_x86_64.manylinux_2_28_x86_64.whl.metadata (2.7 kB)
  Using cached nbformat-5.10.4-py3-none-any.whl.metadata (3.6 kB)
  Using cached pandocfilters-1.5.1-py2.py3-none-any.whl.metadata (9.0 kB)
  Us

## 0. Import Libraries

In [2]:
import pandas as pd
import numpy as np
import os
import shutil

pd.set_option('display.max_columns', 30)
pd.set_option('display.width', 150)

# Folder that represents our "Delta table" storage location
DELTA_TABLE_PATH = '../data/delta_table_customer'
os.makedirs('../data', exist_ok=True)


## 1. Load the Dataset into a Delta Table

We load `customer_master.csv` — this represents the **target** table that already exists
in the lake. In real Delta Lake, this step would be:

```python
df = spark.read.option("header", True).csv("data/customer_master.csv")
df.write.format("delta").mode("overwrite").save("/delta/customer_master")
target = DeltaTable.forPath(spark, "/delta/customer_master")
```

Here, we load it into a pandas DataFrame and persist it as our Delta-table stand-in.

In [3]:
master_raw = pd.read_csv('../data/customer_master.csv')
print(f"Loaded customer_master.csv: {master_raw.shape[0]} rows x {master_raw.shape[1]} columns")
master_raw.head()


Loaded customer_master.csv: 798 rows x 8 columns


,customer_id,customer_name,segment,country,city,state,region,postal_code
0,AA-10315,Alex Avila,Consumer,United States,Minneapolis,Minnesota,Central,55407
1,AA-10375,Allen Armold,Consumer,United States,Mesa,Arizona,West,85204
2,AA-10480,Andrew Allen,Consumer,United States,Concord,North Carolina,South,28027
3,AA-10645,Anna Andreadi,Consumer,United States,Chester,Pennsylvania,East,19013
4,AB-10015,Aaron Bergman,Consumer,United States,Seattle,Washington,West,98103


In [4]:
master_raw.info()


<class 'pandas.DataFrame'>
RangeIndex: 798 entries, 0 to 797
Data columns (total 8 columns):
 #   Column         Non-Null Count  Dtype
---  ------         --------------  -----
 0   customer_id    798 non-null    str  
 1   customer_name  798 non-null    str  
 2   segment        791 non-null    str  
 3   country        798 non-null    str  
 4   city           790 non-null    str  
 5   state          798 non-null    str  
 6   region         798 non-null    str  
 7   postal_code    798 non-null    int64
dtypes: int64(1), str(7)
memory usage: 101.2 KB


## 2. Basic Cleaning (Handle Nulls, Remove Duplicates)

Before writing to the Delta table we clean the raw data:
- Identify and handle missing values in `city` and `segment`
- Remove exact duplicate rows


In [5]:
# Identify missing values
missing = master_raw.isnull().sum()
print("Missing values per column:")
print(missing[missing > 0])


Missing values per column:
segment    7
city       8
dtype: int64


In [6]:
# Handle nulls: fill missing city/segment with 'Unknown' (a defensible default for a
# dimension table where we don't want to silently drop customer records)
master_clean = master_raw.copy()
master_clean['city'] = master_clean['city'].fillna('Unknown')
master_clean['segment'] = master_clean['segment'].fillna('Unknown')

print(f"Missing values after fill: {master_clean.isnull().sum().sum()}")


Missing values after fill: 0


In [7]:
# Remove exact duplicate rows
dup_count = master_clean.duplicated().sum()
print(f"Duplicate rows found: {dup_count}")

master_clean = master_clean.drop_duplicates(subset=['customer_id']).reset_index(drop=True)
print(f"Shape after de-duplication: {master_clean.shape}")


Duplicate rows found: 5
Shape after de-duplication: (793, 8)


In [8]:
# Persist the cleaned target as our Delta table (Parquet-backed)
if os.path.exists(DELTA_TABLE_PATH):
    shutil.rmtree(DELTA_TABLE_PATH)
os.makedirs(DELTA_TABLE_PATH, exist_ok=True)

master_clean.to_parquet(f'{DELTA_TABLE_PATH}/customer_master.parquet', index=False)
print(f"Delta table (customer_master) written with {len(master_clean)} rows.")
master_clean.head()


Delta table (customer_master) written with 793 rows.


,customer_id,customer_name,segment,country,city,state,region,postal_code
0,AA-10315,Alex Avila,Consumer,United States,Minneapolis,Minnesota,Central,55407
1,AA-10375,Allen Armold,Consumer,United States,Mesa,Arizona,West,85204
2,AA-10480,Andrew Allen,Consumer,United States,Concord,North Carolina,South,28027
3,AA-10645,Anna Andreadi,Consumer,United States,Chester,Pennsylvania,East,19013
4,AB-10015,Aaron Bergman,Consumer,United States,Seattle,Washington,West,98103


## 3. Create the Incremental Dataset (Simulated New/Changed Data)

`customer_incremental.csv` simulates a daily CDC (change-data-capture) feed:
- **Updates** — existing customers whose `segment` or `city` changed
- **Inserts** — brand-new customers that don't yet exist in the target table


In [9]:
incremental = pd.read_csv('../data/customer_incremental.csv')
print(f"Loaded customer_incremental.csv: {incremental.shape[0]} rows")

existing_ids = set(master_clean['customer_id'])
incoming_ids = set(incremental['customer_id'])

matched_ids = incoming_ids & existing_ids
new_ids = incoming_ids - existing_ids

print(f"Rows that MATCH existing customers (will UPDATE): {len(matched_ids)}")
print(f"Rows that are brand new (will INSERT):             {len(new_ids)}")

incremental.head()


Loaded customer_incremental.csv: 30 rows
Rows that MATCH existing customers (will UPDATE): 20
Rows that are brand new (will INSERT):             10


,customer_id,customer_name,segment,country,city,state,region,postal_code
0,AH-10030,Aaron Hawkins,Home Office,United States,Georgetown,Pennsylvania,East,19134
1,AW-10930,Arthur Wiediger,Consumer,United States,Madison,Illinois,Central,60505
2,BF-11020,Barry Französisch,Home Office,United States,Springfield,Wisconsin,Central,54302
3,BG-11035,Barry Gonzalez,Home Office,United States,Georgetown,Louisiana,South,71203
4,CA-12310,Christine Abelman,Consumer,United States,Fairview,Ohio,East,45231


## 4. MERGE — SCD Type 1 (Overwrite / Update-in-Place)

SCD Type 1 simply **overwrites** the old attribute values with the new ones — no history
is kept. This is the classic Delta Lake `MERGE INTO` pattern:

```sql
MERGE INTO customer_master AS target
USING customer_incremental AS source
ON target.customer_id = source.customer_id
WHEN MATCHED THEN
  UPDATE SET *
WHEN NOT MATCHED THEN
  INSERT *
```

```python
deltaTable.alias("target").merge(
    incrementalDF.alias("source"),
    "target.customer_id = source.customer_id"
).whenMatchedUpdateAll() \
 .whenNotMatchedInsertAll() \
 .execute()
```

The cell below reproduces this exact MATCHED/NOT-MATCHED logic on our Parquet-backed table.

In [10]:
def scd1_merge(target_df, source_df, key='customer_id'):
    """Replicates Delta Lake's MERGE INTO ... WHEN MATCHED UPDATE SET *
    WHEN NOT MATCHED INSERT * (SCD Type 1) semantics."""
    target_df = target_df.copy().set_index(key)
    source_df = source_df.copy().set_index(key)

    # WHEN MATCHED THEN UPDATE SET * -> overwrite matching rows with source values
    matched_keys = target_df.index.intersection(source_df.index)
    target_df.loc[matched_keys, :] = source_df.loc[matched_keys, :]

    # WHEN NOT MATCHED THEN INSERT * -> append source rows with keys not in target
    new_keys = source_df.index.difference(target_df.index)
    result = pd.concat([target_df, source_df.loc[new_keys, :]])

    return result.reset_index()

scd1_result = scd1_merge(master_clean, incremental)

print(f"Target rows before merge: {len(master_clean)}")
print(f"Source rows (incremental): {len(incremental)}")
print(f"Result rows after SCD1 merge: {len(scd1_result)}")


Target rows before merge: 793
Source rows (incremental): 30
Result rows after SCD1 merge: 803


In [11]:
# Persist SCD1 result as its own Delta table version
scd1_path = f'{DELTA_TABLE_PATH}_scd1'
if os.path.exists(scd1_path):
    shutil.rmtree(scd1_path)
os.makedirs(scd1_path, exist_ok=True)
scd1_result.to_parquet(f'{scd1_path}/customer_master.parquet', index=False)

# Show a sample of updated rows
sample_updated_id = list(matched_ids)[0]
print(f"Example — customer_id = {sample_updated_id}")
print("Before merge:")
display(master_clean[master_clean['customer_id'] == sample_updated_id])
print("After SCD1 merge (overwritten in place, no history kept):")
display(scd1_result[scd1_result['customer_id'] == sample_updated_id])


Example — customer_id = PK-18910


Before merge:


,customer_id,customer_name,segment,country,city,state,region,postal_code
596,PK-18910,Paul Knutson,Home Office,United States,Philadelphia,Pennsylvania,East,19143


After SCD1 merge (overwritten in place, no history kept):


,customer_id,customer_name,segment,country,city,state,region,postal_code
596,PK-18910,Paul Knutson,Corporate,United States,Springfield,Pennsylvania,East,19143


## 5. MERGE — SCD Type 2 (Preserve History)

SCD Type 2 keeps a **full history** of changes: when a matched row's tracked attributes
change, the old row is **expired** (`is_current = false`, `effective_end_date` set) and a
**new version** of the row is inserted as the current record. New customers are inserted
as current with no prior history.

In Delta Lake this is typically expressed as two MERGE statements (or one MERGE with a
staged/unioned source), since a single row can't both update history and insert a new
version in one statement without pre-staging:

```sql
-- Step 1: expire changed records
MERGE INTO customer_scd2 AS target
USING customer_incremental AS source
ON target.customer_id = source.customer_id AND target.is_current = true
WHEN MATCHED AND (
    target.segment <> source.segment OR target.city <> source.city
) THEN
  UPDATE SET target.is_current = false,
             target.effective_end_date = current_date()

-- Step 2: insert new current versions (changed + brand-new customers)
MERGE INTO customer_scd2 AS target
USING staged_new_versions AS source
ON target.customer_id = source.customer_id AND target.is_current = true
WHEN NOT MATCHED THEN
  INSERT (customer_id, customer_name, segment, city, ..., is_current,
          effective_start_date, effective_end_date)
  VALUES (source.customer_id, ..., true, current_date(), NULL)
```

Below, the same two-phase MATCHED/NOT-MATCHED logic is implemented directly.

In [12]:
from datetime import date
TODAY = date.today().isoformat()
TRACKED_COLS = ['segment', 'city']  # attributes whose change triggers a new SCD2 version

def init_scd2(df):
    scd2 = df.copy()
    scd2['is_current'] = True
    scd2['effective_start_date'] = TODAY
    scd2['effective_end_date'] = None
    return scd2

scd2_table = init_scd2(master_clean)
print(f"Initial SCD2 table: {len(scd2_table)} current rows")
scd2_table.head()


Initial SCD2 table: 793 current rows


,customer_id,customer_name,segment,country,city,state,region,postal_code,is_current,effective_start_date,effective_end_date
0,AA-10315,Alex Avila,Consumer,United States,Minneapolis,Minnesota,Central,55407,True,2026-07-05,None
1,AA-10375,Allen Armold,Consumer,United States,Mesa,Arizona,West,85204,True,2026-07-05,None
2,AA-10480,Andrew Allen,Consumer,United States,Concord,North Carolina,South,28027,True,2026-07-05,None
3,AA-10645,Anna Andreadi,Consumer,United States,Chester,Pennsylvania,East,19013,True,2026-07-05,None
4,AB-10015,Aaron Bergman,Consumer,United States,Seattle,Washington,West,98103,True,2026-07-05,None


In [13]:
def scd2_merge(scd2_df, source_df, key='customer_id', tracked_cols=TRACKED_COLS, as_of=TODAY):
    """Replicates a two-phase Delta Lake SCD Type 2 MERGE:
       1) expire current rows whose tracked attributes changed
       2) insert new current versions for changed + brand-new keys
    """
    scd2_df = scd2_df.copy()
    current_mask = scd2_df['is_current'] == True
    current = scd2_df[current_mask].set_index(key)
    source_idx = source_df.set_index(key)

    matched_keys = current.index.intersection(source_idx.index)

    # Determine which matched rows actually changed on tracked columns
    changed_keys = [
        k for k in matched_keys
        if any(current.loc[k, c] != source_idx.loc[k, c] for c in tracked_cols)
    ]

    # ---- WHEN MATCHED AND (attrs changed) THEN UPDATE SET is_current=false, end_date=today
    expire_mask = scd2_df[key].isin(changed_keys) & current_mask
    scd2_df.loc[expire_mask, 'is_current'] = False
    scd2_df.loc[expire_mask, 'effective_end_date'] = as_of

    # ---- WHEN NOT MATCHED THEN INSERT new current version
    new_keys = source_idx.index.difference(current.index)          # brand-new customers
    keys_needing_new_version = list(changed_keys) + list(new_keys)  # changed + new

    new_versions = source_idx.loc[keys_needing_new_version].reset_index()
    new_versions['is_current'] = True
    new_versions['effective_start_date'] = as_of
    new_versions['effective_end_date'] = None

    result = pd.concat([scd2_df, new_versions], ignore_index=True)
    return result, changed_keys, list(new_keys)

scd2_result, changed_keys, brand_new_keys = scd2_merge(scd2_table, incremental)

print(f"Rows expired (history preserved):  {len(changed_keys)}")
print(f"New current versions inserted:     {len(changed_keys) + len(brand_new_keys)}")
print(f"  - from changed customers:        {len(changed_keys)}")
print(f"  - from brand-new customers:      {len(brand_new_keys)}")
print(f"Total SCD2 table rows after merge: {len(scd2_result)}")


Rows expired (history preserved):  20
New current versions inserted:     30
  - from changed customers:        20
  - from brand-new customers:      10
Total SCD2 table rows after merge: 823


In [14]:
# Persist SCD2 result as its own Delta table version
scd2_path = f'{DELTA_TABLE_PATH}_scd2'
if os.path.exists(scd2_path):
    shutil.rmtree(scd2_path)
os.makedirs(scd2_path, exist_ok=True)
scd2_result.to_parquet(f'{scd2_path}/customer_master.parquet', index=False)

# Show the full history for one changed customer: one expired row + one current row
example_id = changed_keys[0]
print(f"Full history for customer_id = {example_id}")
cols_to_show = ['customer_id', 'segment', 'city', 'is_current', 'effective_start_date', 'effective_end_date']
display(scd2_result[scd2_result['customer_id'] == example_id][cols_to_show])


Full history for customer_id = AH-10030


,customer_id,segment,city,is_current,effective_start_date,effective_end_date
27,AH-10030,Corporate,Philadelphia,False,2026-07-05,2026-07-05
793,AH-10030,Home Office,Georgetown,True,2026-07-05,None


## 6. Validate Results (Row Count, Duplicates)

In [15]:
print("=" * 60)
print("VALIDATION — SCD Type 1 result")
print("=" * 60)
expected_scd1_rows = len(master_clean) + len(brand_new_keys)
print(f"Expected rows (target + brand-new): {expected_scd1_rows}")
print(f"Actual rows in SCD1 result:         {len(scd1_result)}")
print(f"Match: {expected_scd1_rows == len(scd1_result)}")

dup_ids_scd1 = scd1_result['customer_id'].duplicated().sum()
print(f"\nDuplicate customer_id values in SCD1 result: {dup_ids_scd1}")
assert dup_ids_scd1 == 0, "SCD1 target should have exactly one row per customer_id"


VALIDATION — SCD Type 1 result
Expected rows (target + brand-new): 803
Actual rows in SCD1 result:         803
Match: True

Duplicate customer_id values in SCD1 result: 0


In [16]:
print("=" * 60)
print("VALIDATION — SCD Type 2 result")
print("=" * 60)

# Exactly one CURRENT row per customer_id
current_rows = scd2_result[scd2_result['is_current'] == True]
dup_current_ids = current_rows['customer_id'].duplicated().sum()
print(f"Duplicate 'current' rows per customer_id: {dup_current_ids}")
assert dup_current_ids == 0, "There should be exactly one current row per customer_id"

# Every customer that existed before or arrived in the incremental feed has a current row
all_expected_ids = set(master_clean['customer_id']) | set(incremental['customer_id'])
missing_current = all_expected_ids - set(current_rows['customer_id'])
print(f"Customers missing a current row: {len(missing_current)}")
assert len(missing_current) == 0

# Historical (expired) rows should never be marked current and must have an end date
expired_rows = scd2_result[scd2_result['is_current'] == False]
bad_expired = expired_rows['effective_end_date'].isnull().sum()
print(f"Expired rows missing an effective_end_date: {bad_expired}")
assert bad_expired == 0

print(f"\nTotal rows in SCD2 table (current + historical): {len(scd2_result)}")
print(f"  - current rows:    {len(current_rows)}")
print(f"  - historical rows: {len(expired_rows)}")
print("\nAll validation checks passed.")


VALIDATION — SCD Type 2 result
Duplicate 'current' rows per customer_id: 0
Customers missing a current row: 0
Expired rows missing an effective_end_date: 0

Total rows in SCD2 table (current + historical): 823
  - current rows:    803
  - historical rows: 20

All validation checks passed.


## 7. Display Final Dataset and Summary

In [17]:
print("Final SCD1 table (current state, no history) — sample:")
display(scd1_result.sort_values('customer_id').head(10))


Final SCD1 table (current state, no history) — sample:


,customer_id,customer_name,segment,country,city,state,region,postal_code
0,AA-10315,Alex Avila,Consumer,United States,Minneapolis,Minnesota,Central,55407
1,AA-10375,Allen Armold,Consumer,United States,Mesa,Arizona,West,85204
2,AA-10480,Andrew Allen,Consumer,United States,Concord,North Carolina,South,28027
3,AA-10645,Anna Andreadi,Consumer,United States,Chester,Pennsylvania,East,19013
4,AB-10015,Aaron Bergman,Consumer,United States,Seattle,Washington,West,98103
5,AB-10060,Adam Bellavance,Home Office,United States,New York City,New York,East,10009
6,AB-10105,Adrian Barton,Consumer,United States,Phoenix,Arizona,West,85023
7,AB-10150,Aimee Bixby,Consumer,United States,Long Beach,New York,East,11561
8,AB-10165,Alan Barnes,Consumer,United States,Unknown,California,West,90036
9,AB-10255,Alejandro Ballentine,Home Office,United States,Lorain,Ohio,East,44052


In [18]:
print("Final SCD2 table (current rows only) — sample:")
display(scd2_result[scd2_result['is_current'] == True]
        .sort_values('customer_id')[['customer_id', 'customer_name', 'segment', 'city',
                                      'is_current', 'effective_start_date']]
        .head(10))


Final SCD2 table (current rows only) — sample:


,customer_id,customer_name,segment,city,is_current,effective_start_date
0,AA-10315,Alex Avila,Consumer,Minneapolis,True,2026-07-05
1,AA-10375,Allen Armold,Consumer,Mesa,True,2026-07-05
2,AA-10480,Andrew Allen,Consumer,Concord,True,2026-07-05
3,AA-10645,Anna Andreadi,Consumer,Chester,True,2026-07-05
4,AB-10015,Aaron Bergman,Consumer,Seattle,True,2026-07-05
5,AB-10060,Adam Bellavance,Home Office,New York City,True,2026-07-05
6,AB-10105,Adrian Barton,Consumer,Phoenix,True,2026-07-05
7,AB-10150,Aimee Bixby,Consumer,Long Beach,True,2026-07-05
8,AB-10165,Alan Barnes,Consumer,Unknown,True,2026-07-05
9,AB-10255,Alejandro Ballentine,Home Office,Lorain,True,2026-07-05


In [19]:
# Save final outputs
scd1_result.to_csv('../data/customer_master_scd1_final.csv', index=False)
scd2_result.to_csv('../data/customer_master_scd2_final.csv', index=False)
print("Saved:")
print(" - data/customer_master_scd1_final.csv")
print(" - data/customer_master_scd2_final.csv")


Saved:
 - data/customer_master_scd1_final.csv
 - data/customer_master_scd2_final.csv


## Summary

| Step | Result |
|---|---|
| Target rows loaded (`customer_master`) | Rows loaded, cleaned of nulls (`city`, `segment` filled) and exact duplicates removed |
| Incremental rows loaded (`customer_incremental`) | Mix of **updates** to existing customers and **brand-new** customer records |
| SCD Type 1 MERGE | Matched rows overwritten in place; unmatched rows inserted. Result has exactly one row per `customer_id`, no history. |
| SCD Type 2 MERGE | Matched rows with changed attributes were **expired** (`is_current=False`, `effective_end_date` set) and a **new current version** inserted; brand-new customers inserted as current. Full change history is preserved. |
| Validation | Row counts reconciled against target + incremental; no duplicate `customer_id` in SCD1; exactly one current row per `customer_id` in SCD2; no expired row left without an end date. |

**Key MERGE INTO concepts demonstrated** (matching Delta Lake's documented semantics):
- `WHEN MATCHED THEN UPDATE` — overwrite matching target rows with source values
- `WHEN NOT MATCHED THEN INSERT` — append source rows that have no match in the target
- Conditional `WHEN MATCHED AND (...)` — only update when specific columns actually changed (used for SCD2 change detection)
- Two-phase MERGE pattern for SCD Type 2 (expire, then insert new version) — the same pattern used in production Delta Lake pipelines when a single-pass MERGE can't both close old versions and open new ones for the same key
